# Entrenamiento de Modelo EfficientNetB0 para Identificación de Plagas

**Universidad Cooperativa de Colombia - Semillero DataTech**

Este notebook implementa el entrenamiento de un modelo de clasificación de plagas agrícolas usando Transfer Learning con EfficientNetB0.

## Características:
- Transfer Learning con EfficientNetB0 (ImageNet)
- Data Augmentation avanzado
- Callbacks: EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
- Fine-tuning opcional
- Métricas de evaluación completas
- Visualización de resultados

## 1. Importar Librerías y Configuración

In [1]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# TensorFlow / Keras
import tensorflow as tf

# Métricas
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
    TensorBoard,
)
from tensorflow.keras.layers import (
    BatchNormalization,
    Dense,
    Dropout,
    GlobalAveragePooling2D,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Importar configuración del proyecto
sys.path.insert(0, str(Path.cwd()))
from config import (
    TRAIN_DIR, VALIDATION_DIR, TEST_DIR, MODELS_DIR,
    MODEL_CONFIG, AUGMENTATION_CONFIG, CALLBACKS_CONFIG,
    ALL_CLASSES, get_model_path, create_directories
)

# Configurar GPU
print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs disponibles: {len(tf.config.list_physical_devices('GPU'))}")

# Reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

TensorFlow version: 2.21.0
GPUs disponibles: 0


## 2. Verificar Dataset

In [ ]:
def count_images(directory):
    """Cuenta las imágenes por clase en un directorio."""
    counts = {}
    if directory.exists():
        for class_dir in directory.iterdir():
            if class_dir.is_dir():
                n_images = len(list(class_dir.glob('*.*')))
                if n_images > 0:
                    counts[class_dir.name] = n_images
    return counts

# Contar imágenes
train_counts = count_images(TRAIN_DIR)
val_counts = count_images(VALIDATION_DIR)
test_counts = count_images(TEST_DIR)

print("=" * 60)
print("ESTADÍSTICAS DEL DATASET")
print("=" * 60)
print(f"\n{'Clase':<35} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 65)

# Obtener clases activas (que tienen imágenes)
active_classes = sorted(set(train_counts.keys()) | set(val_counts.keys()) | set(test_counts.keys()))

for class_name in active_classes:
    t = train_counts.get(class_name, 0)
    v = val_counts.get(class_name, 0)
    ts = test_counts.get(class_name, 0)
    display_name = class_name[:33] + ".." if len(class_name) > 35 else class_name
    print(f"{display_name:<35} {t:>8} {v:>8} {ts:>8}")

print("-" * 65)
print(f"{'TOTAL':<35} {sum(train_counts.values()):>8} {sum(val_counts.values()):>8} {sum(test_counts.values()):>8}")
print(f"\nNúmero de clases: {len(active_classes)}")

## 3. Configurar Generadores de Datos

In [ ]:
# Parámetros
IMG_SIZE = MODEL_CONFIG['input_shape'][:2]  # (224, 224)
BATCH_SIZE = MODEL_CONFIG['batch_size']

print(f"Tamaño de imagen: {IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}")

# Generador con Data Augmentation para entrenamiento
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=AUGMENTATION_CONFIG['rotation_range'],
    width_shift_range=AUGMENTATION_CONFIG['width_shift_range'],
    height_shift_range=AUGMENTATION_CONFIG['height_shift_range'],
    shear_range=AUGMENTATION_CONFIG['shear_range'],
    zoom_range=AUGMENTATION_CONFIG['zoom_range'],
    horizontal_flip=AUGMENTATION_CONFIG['horizontal_flip'],
    brightness_range=AUGMENTATION_CONFIG['brightness_range'],
    fill_mode=AUGMENTATION_CONFIG['fill_mode'],
)

# Generador sin augmentation para validación y test
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# Crear generadores
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
)

validation_generator = val_datagen.flow_from_directory(
    VALIDATION_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

test_generator = val_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False,
)

# Guardar nombres de clases
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)
print(f"\nClases detectadas: {num_classes}")
print(f"Nombres: {class_names}")

## 4. Visualizar Ejemplos de Data Augmentation

In [ ]:
# Visualizar ejemplos de augmentation
def visualize_augmentation(generator, n_images=5):
    """Muestra ejemplos de data augmentation."""
    batch = next(generator)
    images = batch[0][:n_images]
    labels = batch[1][:n_images]
    
    fig, axes = plt.subplots(1, n_images, figsize=(15, 3))
    for i, (img, label) in enumerate(zip(images, labels)):
        axes[i].imshow(img)
        class_idx = np.argmax(label)
        axes[i].set_title(class_names[class_idx][:20])
        axes[i].axis('off')
    
    plt.suptitle('Ejemplos de Data Augmentation', fontsize=14)
    plt.tight_layout()
    plt.show()

visualize_augmentation(train_generator)

## 5. Construir el Modelo

In [ ]:
def build_model(num_classes, input_shape=(224, 224, 3), trainable_base=False):
    """
    Construye el modelo de clasificación basado en EfficientNetB0.
    
    Args:
        num_classes: Número de clases a clasificar.
        input_shape: Forma de la imagen de entrada.
        trainable_base: Si True, las capas del modelo base son entrenables.
    
    Returns:
        Modelo de Keras compilado.
    """
    # Cargar modelo base pre-entrenado
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )
    
    # Congelar capas del modelo base
    base_model.trainable = trainable_base
    
    # Construir capas adicionales
    x = base_model.output
    x = GlobalAveragePooling2D(name='avg_pool')(x)
    
    # Capas densas con Dropout y BatchNormalization
    for i, units in enumerate(MODEL_CONFIG['dense_layers']):
        x = Dense(units, activation='relu', name=f'dense_{i+1}')(x)
        x = BatchNormalization(name=f'bn_{i+1}')(x)
        x = Dropout(MODEL_CONFIG['dropout_rate'], name=f'dropout_{i+1}')(x)
    
    # Capa de salida
    predictions = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    # Crear modelo
    model = Model(inputs=base_model.input, outputs=predictions)
    
    return model, base_model

# Construir modelo
model, base_model = build_model(
    num_classes=num_classes,
    input_shape=MODEL_CONFIG['input_shape'],
    trainable_base=False
)

# Compilar
model.compile(
    optimizer=Adam(learning_rate=MODEL_CONFIG['initial_learning_rate']),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Resumen del modelo
print(f"\nCapas totales: {len(model.layers)}")
print(f"Capas entrenables: {len([l for l in model.layers if l.trainable])}")
print(f"Capas congeladas: {len([l for l in model.layers if not l.trainable])}")
model.summary()

## 6. Configurar Callbacks

In [ ]:
# Crear directorio para modelos si no existe
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Nombre del modelo con timestamp
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_name = f"EfficientNetB0_plagas_{timestamp}.keras"
model_path = MODELS_DIR / model_name

# Callbacks
callbacks = [
    # Early Stopping
    EarlyStopping(
        monitor=CALLBACKS_CONFIG['early_stopping']['monitor'],
        patience=CALLBACKS_CONFIG['early_stopping']['patience'],
        restore_best_weights=CALLBACKS_CONFIG['early_stopping']['restore_best_weights'],
        min_delta=CALLBACKS_CONFIG['early_stopping']['min_delta'],
        verbose=1
    ),
    
    # Reduce Learning Rate
    ReduceLROnPlateau(
        monitor=CALLBACKS_CONFIG['reduce_lr']['monitor'],
        factor=CALLBACKS_CONFIG['reduce_lr']['factor'],
        patience=CALLBACKS_CONFIG['reduce_lr']['patience'],
        min_lr=CALLBACKS_CONFIG['reduce_lr']['min_lr'],
        verbose=1
    ),
    
    # Model Checkpoint
    ModelCheckpoint(
        filepath=str(model_path),
        monitor=CALLBACKS_CONFIG['model_checkpoint']['monitor'],
        save_best_only=CALLBACKS_CONFIG['model_checkpoint']['save_best_only'],
        mode=CALLBACKS_CONFIG['model_checkpoint']['mode'],
        verbose=1
    ),
]

print(f"Modelo se guardará en: {model_path}")
print(f"\nCallbacks configurados:")
for cb in callbacks:
    print(f"  - {cb.__class__.__name__}")

## 7. Entrenamiento - Fase 1 (Capas Congeladas)

In [ ]:
print("=" * 60)
print("FASE 1: ENTRENAMIENTO CON CAPAS BASE CONGELADAS")
print("=" * 60)

# Calcular steps
steps_per_epoch = train_generator.samples // BATCH_SIZE
validation_steps = validation_generator.samples // BATCH_SIZE

print(f"\nSamples de entrenamiento: {train_generator.samples}")
print(f"Steps por época: {steps_per_epoch}")
print(f"Samples de validación: {validation_generator.samples}")
print(f"Validation steps: {validation_steps}")
print(f"Épocas: {MODEL_CONFIG['epochs']}")
print()

# Entrenar
history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=MODEL_CONFIG['epochs'],
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

## 8. Fine-Tuning (Opcional)

In [ ]:
# ============================================================
# FASE 2: FINE-TUNING
# ============================================================
# Estrategia: descongelar solo los últimos bloques de EfficientNetB0
# para adaptar features de alto nivel a nuestro dominio (insectos).
# El LR debe ser ~10-20x menor que Fase 1 para no destruir los pesos.

print("=" * 60)
print("FASE 2: FINE-TUNING - DESCONGELANDO CAPAS SUPERIORES")
print("=" * 60)

# --- 1. Inspeccionar arquitectura de EfficientNetB0 ---
# EfficientNetB0 tiene 7 bloques MBConv (block1a ... block7a).
# Descongelamos solo los últimos 2 bloques + capas de salida (top).
# Las primeras capas detectan bordes/texturas (universales) → se quedan congeladas.

print("\nNombre de capas del base_model (últimas 20):")
for i, layer in enumerate(base_model.layers[-20:], start=len(base_model.layers)-20):
    print(f"  [{i:3d}] {layer.name:<45} trainable={layer.trainable}")

# --- 2. Definir punto de descongelamiento ---
# Buscamos la primera capa del bloque 6 para descongelar desde ahí.
UNFREEZE_FROM_BLOCK = "block6"   # descongelar bloque 6 y 7 + top

base_model.trainable = True  # primero habilitar todo

frozen_count = 0
unfrozen_count = 0
for layer in base_model.layers:
    # Congelar todo lo anterior al bloque objetivo
    if UNFREEZE_FROM_BLOCK in layer.name:
        # A partir de aquí dejamos entrenable
        pass
    else:
        layer.trainable = False
        frozen_count += 1
        continue
    unfrozen_count += 1

print(f"\nCapas congeladas  (base_model): {frozen_count}")
print(f"Capas entrenables (base_model): {unfrozen_count}")
print(f"Capas entrenables totales     : {len([l for l in model.layers if l.trainable])}")

# --- 3. Recompilar con LR muy bajo ---
# CRÍTICO: usar LR bajo (1e-5) para no destruir los pesos pre-entrenados.
FINE_TUNE_LR = MODEL_CONFIG.get('fine_tune_learning_rate', 1e-5)

model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"\nLearning rate Fase 2: {FINE_TUNE_LR}")

# --- 4. Callbacks para Fase 2 ---
model_name_ft = f"EfficientNetB0_plagas_finetuned_{timestamp}.keras"
model_path_ft = MODELS_DIR / model_name_ft

callbacks_ft = [
    EarlyStopping(
        monitor='val_loss',
        patience=7,                  # más paciencia que Fase 1
        restore_best_weights=True,
        min_delta=0.001,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        filepath=str(model_path_ft),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
]

# --- 5. Entrenar Fase 2 ---
# initial_epoch continúa desde donde terminó Fase 1 (para que la gráfica sea continua)
fine_tune_epochs = MODEL_CONFIG.get('fine_tune_epochs', 30)
fase1_epochs = len(history.history['loss'])   # épocas reales que corrió Fase 1

print(f"\nÉpocas Fase 1 completadas : {fase1_epochs}")
print(f"Épocas adicionales Fase 2 : {fine_tune_epochs}")
print(f"Época final               : {fase1_epochs + fine_tune_epochs}")
print()

history_ft = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=fase1_epochs + fine_tune_epochs,
    initial_epoch=fase1_epochs,
    validation_data=validation_generator,
    validation_steps=validation_steps,
    callbacks=callbacks_ft,
    verbose=1
)

# --- 6. Combinar historiales (para graficar curva completa) ---
for key in history.history.keys():
    if key in history_ft.history:
        history.history[key].extend(history_ft.history[key])

# Guardar el epoch de inicio del fine-tuning para marcarlo en la gráfica
FINE_TUNE_START_EPOCH = fase1_epochs

print(f"\nModelo fine-tuned guardado en: {model_path_ft}")
print(f"Mejor val_accuracy Fase 2: {max(history_ft.history['val_accuracy']):.4f}")


## 9. Visualizar Historial de Entrenamiento

In [ ]:
def plot_training_history(history, fine_tune_start=None):
    """
    Grafica el historial de entrenamiento completo.
    Si fine_tune_start se provee, dibuja una línea vertical
    indicando dónde comenzó el fine-tuning.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs_range = range(1, len(history.history['accuracy']) + 1)

    for ax, metric, title, ylabel in [
        (axes[0], 'accuracy', 'Accuracy', 'Accuracy'),
        (axes[1], 'loss',     'Loss',     'Loss'),
    ]:
        ax.plot(epochs_range, history.history[metric],
                label='Train', linewidth=2)
        ax.plot(epochs_range, history.history[f'val_{metric}'],
                label='Validation', linewidth=2)

        # Línea vertical en inicio de fine-tuning
        if fine_tune_start is not None:
            ax.axvline(x=fine_tune_start, color='gray',
                       linestyle='--', linewidth=1.5,
                       label=f'Fine-tuning (época {fine_tune_start})')

        ax.set_title(title, fontsize=14)
        ax.set_xlabel('Época')
        ax.set_ylabel(ylabel)
        ax.legend()
        ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(MODELS_DIR / f'training_history_{timestamp}.png', dpi=150)
    plt.show()

    print("\nMétricas finales:")
    print(f"  Train Accuracy : {history.history['accuracy'][-1]:.4f}")
    print(f"  Val Accuracy   : {history.history['val_accuracy'][-1]:.4f}")
    print(f"  Train Loss     : {history.history['loss'][-1]:.4f}")
    print(f"  Val Loss       : {history.history['val_loss'][-1]:.4f}")

# FINE_TUNE_START_EPOCH queda definido en la celda de Fase 2.
# Si no se corrió fine-tuning, se pasa None.
_ft_start = FINE_TUNE_START_EPOCH if 'FINE_TUNE_START_EPOCH' in dir() else None
plot_training_history(history, fine_tune_start=_ft_start)


## 10. Evaluación en Conjunto de Test

In [ ]:
print("=" * 60)
print("EVALUACIÓN EN CONJUNTO DE TEST")
print("=" * 60)

# Evaluar modelo
test_generator.reset()
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)

print(f"\nResultados en Test:")
print(f"  Loss:     {test_loss:.4f}")
print(f"  Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 11. Matriz de Confusión y Reporte de Clasificación

In [ ]:
# Obtener predicciones
test_generator.reset()
predictions = model.predict(test_generator, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

# Matriz de confusión
cm = confusion_matrix(y_true, y_pred)

# Visualizar matriz de confusión
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.title('Matriz de Confusión', fontsize=14)
plt.xlabel('Predicción')
plt.ylabel('Real')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(MODELS_DIR / f'confusion_matrix_{timestamp}.png', dpi=150)
plt.show()

# Reporte de clasificación
print("\n" + "=" * 60)
print("REPORTE DE CLASIFICACIÓN")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=class_names))

## 12. Guardar Modelo Final

In [ ]:
# Guardar modelo final
final_model_path = MODELS_DIR / 'EfficientNetB0_plagas_final.keras'
model.save(final_model_path)

print(f"\nModelo guardado en: {final_model_path}")
print(f"\nArchivos generados:")
for f in MODELS_DIR.glob('*'):
    print(f"  - {f.name}")

## 13. Función de Predicción

In [ ]:
from tensorflow.keras.preprocessing import image
from PIL import Image

def predict_image(img_path, model, class_names, target_size=(224, 224)):
    """
    Predice la clase de una imagen.
    
    Args:
        img_path: Ruta a la imagen.
        model: Modelo entrenado.
        class_names: Lista de nombres de clases.
        target_size: Tamaño de imagen esperado por el modelo.
    
    Returns:
        Diccionario con predicciones y probabilidades.
    """
    # Cargar y preprocesar imagen
    img = Image.open(img_path)
    img = img.resize(target_size, Image.LANCZOS)
    img = img.convert('RGB')
    
    # Convertir a array
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = img_array / 255.0  # Normalizar
    
    # Predecir
    predictions = model.predict(img_array, verbose=0)[0]
    
    # Obtener clase predicha
    predicted_idx = np.argmax(predictions)
    predicted_class = class_names[predicted_idx]
    confidence = predictions[predicted_idx]
    
    # Crear resultado
    result = {
        'predicted_class': predicted_class,
        'confidence': float(confidence),
        'probabilities': {class_names[i]: float(p) for i, p in enumerate(predictions)}
    }
    
    return result

# Ejemplo de uso (descomentar para probar)
# result = predict_image('ruta/a/imagen.jpg', model, class_names)
# print(f"Predicción: {result['predicted_class']}")
# print(f"Confianza: {result['confidence']:.2%}")

## 14. Resumen Final

In [ ]:
print("=" * 60)
print("RESUMEN DEL ENTRENAMIENTO")
print("=" * 60)
print(f"\nModelo: EfficientNetB0")
print(f"Clases: {num_classes}")
print(f"Imágenes de entrenamiento: {train_generator.samples}")
print(f"Imágenes de validación: {validation_generator.samples}")
print(f"Imágenes de test: {test_generator.samples}")
print(f"\nMejor accuracy en validación: {max(history.history['val_accuracy']):.4f}")
print(f"Accuracy en test: {test_accuracy:.4f}")
print(f"\nModelo guardado en: {final_model_path}")
print("=" * 60)